# o que achou dessa aplicacao ?

Classifique texto em **Português** ou **Inglês** como BOM/RUIM com as BiLSTMs do notebook FASE7/RNN. O gráfico final mostra a probabilidade positiva depois de cada palavra. O corte é 50%; o último ponto da curva é a probabilidade usada na classificação.

O modelo português B2W foi preservado; a rede inglesa foi treinada separadamente com IMDB. Não há notas manuais, histórico ou gravação dos textos. Use o kernel Python (FASE7).

In [ ]:
from pathlib import Path
import sys, json
base = Path.cwd().resolve()
RAIZ = next((p for a in (base, *base.parents)
             for p in (a, a / "ANO1/Sompo-AgroPredict-main/Sompo-AgroPredict-main")
             if (p / "src/agropredict/feedback_modelo.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra o Jupyter dentro do projeto FIAP/Sompo.")
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "src/dashboard"))
from agropredict.feedback_modelo import AnalisadorFeedback
from paginas.feedback import grafico_leitura
IDIOMA = "pt"  # pt = Português; en = Inglês
nome = "feedback_bilstm.pt" if IDIOMA == "pt" else "feedback_bilstm_en.pt"
analisador = AnalisadorFeedback(RAIZ / "src/modelo" / nome, IDIOMA)
print("Dispositivo:", analisador.device)
print(json.dumps(analisador.metricas, ensure_ascii=False, indent=2))


## Classificar e acompanhar a leitura

A rede e o vocabulário correspondem ao idioma selecionado. Os prefixos são avaliados em lotes. Português lê até 119 palavras, inglês até 199, além do token inicial. A probabilidade do texto completo considerado corresponde ao último ponto.

In [ ]:
TEXTO = ("Gostei muito do projeto, excelente e fácil de usar." if IDIOMA == "pt"
         else "This is an excellent wonderful application, I love it.")
resultado = analisador.classificar(TEXTO)
print(resultado["classificacao"])
print(f"BOM: {resultado['prob_bom']:.1%} | RUIM: {resultado['prob_ruim']:.1%}")
assert resultado["leitura"][-1] == resultado["prob_bom"]


In [ ]:
from IPython.display import HTML, display
from paginas.feedback import html_raio_x, diagnostico_tecnico
display(HTML(html_raio_x(resultado)))
print(diagnostico_tecnico(resultado))


In [ ]:
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
from IPython.display import display
display(grafico_leitura(resultado))


## Treino offline opcional do inglês

Os dados IMDB já estão em FASE7/RNN/data. A preparação mantém os IDs deslocados em 3, token inicial 1, desconhecido 2, vocabulário 10.000 e sequências de 200 posições. O treino usa 22.500 exemplos, validação 2.500 e teste oficial separado de 25.000. Três épocas, escolha de pesos por perda de validação. A execução abaixo só treina o inglês, preservando o arquivo português.

In [ ]:
RETREINAR_INGLES = False
if RETREINAR_INGLES:
    import subprocess
    subprocess.run([sys.executable, str(RAIZ / "scripts/treinar_feedback_ingles.py")], check=True)


## Streamlit

A página oferece o seletor Português/Inglês independentemente do idioma da interface. O título é exatamente "o que achou dessa aplicacao ?". Ao alterar texto ou idioma, o resultado antigo é limpo. Não há uso do modelo de outro idioma quando o selecionado está indisponível.

As métricas B2W e IMDB refletem avaliações de produtos e filmes, sem validação específica em comentários do AgroPredict. As probabilidades não são confiança calibrada. Consulte document/feedback_projeto.md.